# Анализ производительности реализаций кеша
Сравнение `IMemoryCache`, `IDistributedCache` (Redis, Valkey, Garnet) и `HybridCache`.

In [6]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import re

CSV_PATH = "../RequestMonitoring.Benchmarks/bin/Release/net10.0/BenchmarkDotNet.Artifacts/results/RequestMonitoring.Benchmarks.CacheBenchmark-report.csv"

df = pd.read_csv(CSV_PATH, sep=";")
df = df[["Method", "Mean", "Error", "StdDev", "Allocated"]].copy()

def parse_ns(val):
    # Убираем кавычки, запятые, единицы измерения (ns, μs, ms и т.д.)
    val = str(val).replace('"', '').replace(',', '').strip()
    val = re.sub(r'[a-zA-Z\s]', '', val)
    return float(val)

for col in ["Mean", "Error", "StdDev"]:
    df[col] = df[col].apply(parse_ns)

df["Allocated_B"] = df["Allocated"].astype(str).str.replace(' B', '').str.replace(',', '').str.strip().astype(float)
df["Operation"] = df["Method"].apply(lambda x: "Get" if "Get" in x else "Set")
df["Cache"] = df["Method"].str.strip("'")
df["Mean_us"] = df["Mean"] / 1000
df["Error_us"] = df["Error"] / 1000

df.head(10)

,Method,Mean,Error,StdDev,Allocated,Allocated_B,Operation,Cache,Mean_us,Error_us
0,'IMemoryCache Get',29.59,0.486,0.455,0 B,0.0,Get,IMemoryCache Get,0.02959,0.000486
1,'IMemoryCache Set',66.00,1.011,0.896,104 B,104.0,Set,IMemoryCache Set,0.06600,0.001011
2,'HybridCache Get',8595.88,803.987,2332.511,72 B,72.0,Get,HybridCache Get,8.59588,0.803987
3,'IDistributedCache Valkey Get',373170.13,7308.012,10938.290,1264 B,1264.0,Get,IDistributedCache Valkey Get,373.17013,7.308012
4,'IDistributedCache Valkey Set',373171.22,7298.414,10231.358,1496 B,1496.0,Set,IDistributedCache Valkey Set,373.17122,7.298414
5,'IDistributedCache Redis Get',377640.96,7375.064,7243.299,1264 B,1264.0,Get,IDistributedCache Redis Get,377.64096,7.375064
6,'HybridCache Set',380510.24,7269.566,8653.902,2264 B,2264.0,Set,HybridCache Set,380.51024,7.269566
7,'IDistributedCache Redis Set',383938.05,7280.633,7476.674,1496 B,1496.0,Set,IDistributedCache Redis Set,383.93805,7.280633
8,'IDistributedCache Garnet Set',414984.92,6754.543,5987.726,1496 B,1496.0,Set,IDistributedCache Garnet Set,414.98492,6.754543
9,'IDistributedCache Garnet Get',415580.79,8191.208,10359.256,1264 B,1264.0,Get,IDistributedCache Garnet Get,415.58079,8.191208


In [7]:
# График 1: Mean latency Get vs Set (логарифмическая шкала)

fig = make_subplots(rows=1, cols=2, subplot_titles=("Get", "Set"))
colors = px.colors.qualitative.Plotly

for i, op in enumerate(["Get", "Set"]):
    subset = df[df["Operation"] == op].sort_values("Mean")
    fig.add_trace(
        go.Bar(
            x=subset["Cache"],
            y=subset["Mean_us"],
            error_y=dict(type="data", array=subset["Error_us"].tolist()),
            name=op,
            marker_color=colors[:len(subset)],
            showlegend=False
        ),
        row=1, col=i+1
    )

fig.update_yaxes(title_text="Среднее время (μs)", type="log")
fig.update_xaxes(tickangle=-30)
fig.update_layout(title_text="Latency реализаций кеша (логарифмическая шкала)", height=500)
fig.show()

In [8]:
# График 2: Сравнение только distributed кешей (Redis, Valkey, Garnet)

distributed = df[df["Cache"].str.contains("Redis|Valkey|Garnet")].copy()

fig2 = px.bar(
    distributed.sort_values("Mean"),
    x="Cache",
    y="Mean_us",
    error_y="Error_us",
    color="Operation",
    barmode="group",
    title="Сравнение Redis / Valkey / Garnet",
    labels={"Mean_us": "Среднее время (μs)", "Cache": ""}
)
fig2.update_xaxes(tickangle=-30)
fig2.show()

In [9]:
# График 3: Аллокации памяти

fig3 = px.bar(
    df.sort_values("Allocated_B"),
    x="Cache",
    y="Allocated_B",
    color="Operation",
    barmode="group",
    title="Аллокации памяти на операцию",
    labels={"Allocated_B": "Байт", "Cache": ""}
)
fig3.update_xaxes(tickangle=-30)
fig3.show()

In [10]:
# Итоговая таблица

summary = df[["Cache", "Operation", "Mean_us", "Error_us", "Allocated_B"]].copy()
summary.columns = ["Реализация", "Операция", "Среднее (μs)", "Погрешность (μs)", "Аллокации (B)"]
summary = summary.sort_values("Среднее (μs)")
summary.style.background_gradient(subset=["Среднее (μs)"], cmap="RdYlGn_r")

,Реализация,Операция,Среднее (μs),Погрешность (μs),Аллокации (B)
0,IMemoryCache Get,Get,0.029590,0.000486,0.000000
1,IMemoryCache Set,Set,0.066000,0.001011,104.000000
2,HybridCache Get,Get,8.595880,0.803987,72.000000
3,IDistributedCache Valkey Get,Get,373.170130,7.308012,1264.000000
4,IDistributedCache Valkey Set,Set,373.171220,7.298414,1496.000000
5,IDistributedCache Redis Get,Get,377.640960,7.375064,1264.000000
6,HybridCache Set,Set,380.510240,7.269566,2264.000000
7,IDistributedCache Redis Set,Set,383.938050,7.280633,1496.000000
8,IDistributedCache Garnet Set,Set,414.984920,6.754543,1496.000000
9,IDistributedCache Garnet Get,Get,415.580790,8.191208,1264.000000
